# Stage 1 · RL Foundations — SOLUTION
### Topics: MDPs · Bellman Equations · Policy Gradient Theorem · Advantage & GAE · REINFORCE · A2C · PPO


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import List, Tuple, Optional
from dataclasses import dataclass, field


---
## 1 · Markov Decision Process & Discounted Returns

An MDP is a tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:
- $\mathcal{S}$ — state space; $\mathcal{A}$ — action space
- $P(s'|s,a)$ — transition dynamics
- $R(s,a)$ — reward function
- $\gamma \in [0,1)$ — discount factor (controls myopia)

### Discounted return from timestep t
$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$

**Monte Carlo return:** compute $G_t$ backwards from the end of the episode — exact but high variance.

### Bellman Equations
Value function: $V^\pi(s) = \mathbb{E}_\pi[G_t | s_t = s]$  
Action-value:   $Q^\pi(s,a) = \mathbb{E}_\pi[G_t | s_t=s, a_t=a]$  

Bellman consistency:
$$V^\pi(s) = \sum_a \pi(a|s)\sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^\pi(s')]$$

**Advantage:** $A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$ — how much better is action $a$ vs the average?


In [9]:
def compute_returns(rewards: List[float], gamma: float) -> List[float]:
    """
    Monte Carlo discounted returns: G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...
    Computed efficiently in one backward pass: G_T = r_T, G_t = r_t + γ*G_{t+1}
    """
    T       = len(rewards)
    returns = [0.0] * T
    G       = 0.0
    for t in reversed(range(T)):
        G          = rewards[t] + gamma * G
        returns[t] = G
    return returns


def normalize_returns(returns: List[float], eps: float = 1e-8) -> List[float]:
    """Standardise returns to zero mean, unit variance (reduces variance in policy gradient)."""
    r = np.array(returns, dtype=np.float32)
    return list((r - r.mean()) / (r.std() + eps))


# ── Sanity checks ─────────────────────────────────────────────────────────
# γ=0: G_t = r_t (myopic)
rewards = [1.0, 2.0, 3.0]
returns_myopic = compute_returns(rewards, gamma=0.0)
assert returns_myopic == [1.0, 2.0, 3.0], returns_myopic

# γ=1: G_t = sum of all future rewards
returns_nodiscount = compute_returns(rewards, gamma=1.0)
assert returns_nodiscount == [6.0, 5.0, 3.0], returns_nodiscount

# γ=0.9: verify recursion manually
r = [1.0, 2.0, 3.0]
g = compute_returns(r, gamma=0.9)
assert abs(g[0] - (1 + 0.9*2 + 0.81*3)) < 1e-5

normed = normalize_returns(g)
assert abs(np.mean(normed)) < 1e-5, "mean should be ~0"
assert abs(np.std(normed)  - 1.0) < 1e-5, "std should be ~1"

print("compute_returns    ✓")
print("normalize_returns  ✓")
print(f"  γ=0.9 returns: {[round(x,3) for x in g]}")
print(f"  normalized:    {[round(x,3) for x in normed]}")


compute_returns    ✓
normalize_returns  ✓
  γ=0.9 returns: [5.23, 4.7, 3.0]
  normalized:    [0.967, 0.41, -1.377]


---
## 2 · Policy Gradient Theorem & REINFORCE

### Policy Gradient Theorem (Williams, 1992)
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_t G_t \nabla_\theta \log \pi_\theta(a_t|s_t)\right]$$

We minimise the **negative** expected return:
$$\mathcal{L}_{PG} = -\sum_t G_t \log \pi_\theta(a_t|s_t)$$

**Variance reduction:** replacing $G_t$ with the **advantage** $A_t = G_t - b(s_t)$ where $b$ is a baseline (e.g., $V(s_t)$) leaves the gradient unbiased but reduces variance dramatically.

### REINFORCE with baseline algorithm
```
for each episode:
    collect trajectory τ = {s₀,a₀,r₀, ..., s_T,a_T,r_T}
    compute G_t for all t  (Monte Carlo)
    compute A_t = G_t - V(s_t)          # baseline subtraction
    loss = -∑_t A_t · log π(a_t|s_t)
    loss.backward(); optimizer.step()
```

### Policy networks
A discrete policy outputs a **categorical distribution** over actions: `logits → softmax → sample`.  
Key: `log_prob = log_softmax(logits)[action]` — the log-prob of the *chosen* action.


In [10]:
class DiscretePolicy(nn.Module):
    """Simple MLP policy for discrete action spaces."""
    def __init__(self, obs_dim: int, act_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )

    def forward(self, obs: torch.Tensor) -> torch.distributions.Categorical:
        logits = self.net(obs)
        return torch.distributions.Categorical(logits=logits)

    def act(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample action and return (action, log_prob)."""
        dist    = self.forward(obs)
        action  = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob


def reinforce_loss(
    log_probs: torch.Tensor,   # (T,) — log π(a_t|s_t) for each timestep
    advantages: torch.Tensor,  # (T,) — G_t or A_t, must be detached
) -> torch.Tensor:
    """REINFORCE: L = -mean(A_t * log π(a_t|s_t))"""
    assert not advantages.requires_grad, "advantages must be detached"
    return -(advantages * log_probs).mean()


def entropy_bonus(dist: torch.distributions.Categorical, coef: float = 0.01) -> torch.Tensor:
    """
    Entropy H(π) = -∑ π(a) log π(a).
    Add to loss as -coef*H to encourage exploration.
    High entropy = uniform = exploring; low entropy = deterministic = exploiting.
    """
    return -coef * dist.entropy().mean()


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
obs_dim, act_dim = 4, 2    # e.g., CartPole

policy = DiscretePolicy(obs_dim, act_dim)
obs    = torch.randn(8, obs_dim)   # batch of 8 observations
dist   = policy(obs)

actions, lp = policy.act(obs)
assert actions.shape == (8,),  f"actions shape: {actions.shape}"
assert lp.shape     == (8,),   f"log_prob shape: {lp.shape}"
assert (lp <= 0).all(),         "log-probs must be ≤ 0"

adv    = torch.tensor([1.0, -0.5, 0.3, 0.8, -1.0, 0.2, 0.4, -0.3])
loss   = reinforce_loss(lp, adv)
assert loss.shape == (), "loss must be scalar"

ent    = entropy_bonus(dist)
print(f"DiscretePolicy ✓  actions: {actions.tolist()}")
print(f"reinforce_loss ✓  loss={loss.item():.4f}")
print(f"entropy_bonus  ✓  H={dist.entropy().mean().item():.4f}")


DiscretePolicy ✓  actions: [1, 0, 0, 1, 0, 1, 0, 0]
reinforce_loss ✓  loss=0.1152
entropy_bonus  ✓  H=0.6901


---
## 3 · Value Function, TD Error & Advantage

### Value network
A separate MLP $V_\phi(s) \approx V^\pi(s)$ trained via regression on Monte Carlo returns:
$$\mathcal{L}_V = \frac{1}{T}\sum_t (V_\phi(s_t) - G_t)^2$$

### TD(0) one-step error (temporal difference)
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is a biased but **low-variance** advantage estimate.

### Generalised Advantage Estimation (GAE, Schulman 2015)
Interpolates between Monte Carlo (low bias, high variance) and TD(0) (high bias, low variance):

$$\hat{A}_t^{GAE(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

- $\lambda=1$: reduces to Monte Carlo advantage (high variance)
- $\lambda=0$: reduces to TD(0) error (high bias)
- $\lambda \approx 0.95$: sweet spot used in PPO and most modern algorithms

**Efficient computation** (backward pass, exactly like returns):
$$\hat{A}_T = \delta_T, \quad \hat{A}_t = \delta_t + \gamma\lambda \hat{A}_{t+1}$$


In [11]:
class ValueNetwork(nn.Module):
    """Critic: obs → scalar value estimate V(s)."""
    def __init__(self, obs_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs).squeeze(-1)   # (B,)


def value_loss(values: torch.Tensor, returns: torch.Tensor) -> torch.Tensor:
    """MSE between predicted values and Monte Carlo returns."""
    return F.mse_loss(values, returns)


def compute_td_errors(
    rewards: torch.Tensor,    # (T,)
    values:  torch.Tensor,    # (T,)   — V(s_t), detached
    next_values: torch.Tensor,# (T,)   — V(s_{t+1}), 0 at terminal
    gamma: float,
) -> torch.Tensor:            # (T,)
    """δ_t = r_t + γ·V(s_{t+1}) - V(s_t)"""
    return rewards + gamma * next_values - values


def compute_gae(
    rewards:     torch.Tensor,  # (T,)
    values:      torch.Tensor,  # (T,)  — detached
    next_values: torch.Tensor,  # (T,)  — V(s_{t+1}), 0 at terminal
    gamma: float,
    lam:   float,
) -> torch.Tensor:              # (T,)  — GAE advantages
    """
    GAE: Â_t = δ_t + (γλ)·δ_{t+1} + (γλ)²·δ_{t+2} + ...
    Computed via backward pass: A_T = δ_T, A_t = δ_t + γλ·A_{t+1}
    """
    deltas = compute_td_errors(rewards, values, next_values, gamma)  # (T,)
    T      = len(rewards)
    adv    = torch.zeros(T)
    gae    = 0.0
    for t in reversed(range(T)):
        gae    = deltas[t].item() + gamma * lam * gae
        adv[t] = gae
    return adv


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
T, obs_dim = 10, 4
critic = ValueNetwork(obs_dim)
obs    = torch.randn(T, obs_dim)
vals   = critic(obs)
assert vals.shape == (T,), f"Value shape: {vals.shape}"

# TD errors: if V is perfect (V(s) = r + γV(s')), errors should be 0
rewards     = torch.ones(T)
next_vals   = torch.zeros(T)  # terminal
next_vals[:T-1] = vals[1:].detach()
td = compute_td_errors(rewards, vals.detach(), next_vals.detach(), gamma=0.99)
assert td.shape == (T,)

# GAE with λ=0 should equal TD errors
gae_td0 = compute_gae(rewards, vals.detach(), next_vals.detach(), gamma=0.99, lam=0.0)
assert torch.allclose(gae_td0, td, atol=1e-5), "GAE(λ=0) should equal TD error"

print(f"ValueNetwork ✓  values: {vals[:3].tolist()}")
print(f"compute_gae  ✓  GAE(λ=0) == TD error: {torch.allclose(gae_td0, td, atol=1e-5)}")
print(f"  TD errors  (first 3): {td[:3].tolist()}")
print(f"  GAE(λ=0.95)(first 3): {compute_gae(rewards, vals.detach(), next_vals.detach(), 0.99, 0.95)[:3].tolist()}")


ValueNetwork ✓  values: [-0.009031549096107483, -0.21818965673446655, 0.007536694407463074]
compute_gae  ✓  GAE(λ=0) == TD error: True
  TD errors  (first 3): [0.7930237650871277, 1.2256510257720947, 1.037315011024475]
  GAE(λ=0.95)(first 3): [7.686784744262695, 7.329889297485352, 6.490418434143066]


---
## 4 · Proximal Policy Optimization (PPO)

PPO (Schulman et al., 2017) is the direct ancestor of GRPO and the dominant deep RL algorithm.

### Key insight: trust region without second-order optimisation
Instead of constraining the KL divergence explicitly (TRPO), PPO clips the probability ratio:

$$\rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$$

$$\mathcal{L}^{CLIP} = -\mathbb{E}_t\left[\min\left(\rho_t A_t,\ \text{clip}(\rho_t, 1-\varepsilon, 1+\varepsilon)A_t\right)\right]$$

### Why clipping works
- If $A_t > 0$ (good action): cap ratio at $1+\varepsilon$ — don't increase probability too aggressively
- If $A_t < 0$ (bad action): floor ratio at $1-\varepsilon$ — don't decrease probability too aggressively
- The `min` ensures we take the **pessimistic** bound — always conservative

### Full PPO loss
$$\mathcal{L} = \mathcal{L}^{CLIP} + c_1 \mathcal{L}^V - c_2 H(\pi_\theta)$$

where $c_1 \approx 0.5$ (value coeff), $c_2 \approx 0.01$ (entropy coeff).

### PPO training loop
```
Collect N steps of experience with π_θ_old
Compute GAE advantages and returns
For K epochs:
    For each minibatch:
        Compute ρ, clip loss, value loss, entropy
        Update θ with gradient descent
```


In [12]:
def ppo_clip_loss(
    log_probs_new: torch.Tensor,   # (B,) — log π_θ(a|s)
    log_probs_old: torch.Tensor,   # (B,) — log π_θ_old(a|s), detached
    advantages:    torch.Tensor,   # (B,) — detached GAE advantages
    epsilon: float = 0.2,
) -> torch.Tensor:
    """PPO clipped surrogate loss."""
    ratio = torch.exp(log_probs_new - log_probs_old.detach())
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    return -torch.min(surr1, surr2).mean()


def ppo_loss(
    log_probs_new: torch.Tensor,   # (B,)
    log_probs_old: torch.Tensor,   # (B,)  detached
    values:        torch.Tensor,   # (B,)  V_φ(s_t), with grad
    returns:       torch.Tensor,   # (B,)  MC returns, detached
    advantages:    torch.Tensor,   # (B,)  GAE, detached
    entropy:       torch.Tensor,   # ()    H(π_θ)
    epsilon:     float = 0.2,
    vf_coef:     float = 0.5,
    ent_coef:    float = 0.01,
) -> Tuple[torch.Tensor, dict]:
    """
    Full PPO loss = clip_loss + vf_coef*value_loss - ent_coef*entropy
    """
    clip = ppo_clip_loss(log_probs_new, log_probs_old, advantages, epsilon)
    vf   = vf_coef  * value_loss(values, returns.detach())
    ent  = ent_coef * entropy
    total = clip + vf - ent
    return total, {"clip": clip.item(), "vf": vf.item(), "ent": entropy.item()}


def compute_clip_fraction(
    log_probs_new: torch.Tensor,
    log_probs_old: torch.Tensor,
    epsilon: float = 0.2,
) -> float:
    """
    Fraction of samples where ratio is clipped.
    High clip fraction (> 0.3) → step size too large → reduce lr or epsilon.
    """
    ratio  = torch.exp((log_probs_new - log_probs_old).detach())
    clipped = ((ratio < 1 - epsilon) | (ratio > 1 + epsilon)).float()
    return clipped.mean().item()


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 16
lp_new = torch.randn(B, requires_grad=True)
lp_old = lp_new.detach().clone()
adv    = torch.randn(B)

# If ratio = 1 (same policy), clipped loss == -mean(A)
loss_same = ppo_clip_loss(lp_new, lp_old, adv)
expected  = -adv.mean()
assert abs(loss_same.item() - expected.item()) < 1e-5,     f"ratio=1 loss {loss_same.item()} != {expected.item()}"

# Clip fraction = 0 when policies are identical
cf = compute_clip_fraction(lp_new, lp_old)
assert cf == 0.0, f"Expected 0 clip fraction, got {cf}"

# Clip fraction increases when policy drifts
lp_drifted = lp_old + 1.0   # large shift → ratio >> 1
cf_high = compute_clip_fraction(lp_drifted, lp_old)
assert cf_high > 0.5, f"Expected high clip fraction, got {cf_high}"

vals    = torch.randn(B, requires_grad=True)
returns = torch.randn(B)
dist    = torch.distributions.Categorical(logits=torch.randn(B, 4))
total, info = ppo_loss(lp_new, lp_old, vals, returns, adv, dist.entropy().mean())
total.backward()

print(f"ppo_clip_loss        ✓  loss (ratio=1) = {loss_same.item():.4f}")
print(f"compute_clip_fraction ✓  clip_frac (same) = {cf:.2f}, (drifted) = {cf_high:.2f}")
print(f"ppo_loss              ✓  {info}")


ppo_clip_loss        ✓  loss (ratio=1) = -0.1325
compute_clip_fraction ✓  clip_frac (same) = 0.00, (drifted) = 1.00
ppo_loss              ✓  {'clip': -0.13250373303890228, 'vf': 1.4941009283065796, 'ent': 1.093965768814087}


---
## 5 · A2C Training Loop on a Toy MDP

We put everything together in an **Actor-Critic (A2C)** loop on a simple GridWorld MDP.

### A2C vs REINFORCE
| | REINFORCE | A2C |
|---|---|---|
| Advantage | MC returns | TD error or GAE |
| Variance | High | Lower |
| Bias | None (MC) | Some (bootstrapping) |
| Update | Per episode | Per step or batch |

### Toy MDP: Linear Chain
- States: 0, 1, ..., N-1
- Actions: 0 (left), 1 (right)
- Reward: +1 if reaching state N-1, else 0
- Episode ends at states 0 or N-1
- Optimal policy: always go right

We test that training reduces loss and improves the policy's preference for right actions.


In [13]:
class LinearChainMDP:
    """
    Simple chain MDP: states 0..N-1, actions {0=left, 1=right}.
    Reward +1 only at terminal state N-1. Episode ends at 0 or N-1.
    """
    def __init__(self, N: int = 6):
        self.N     = N
        self.state = N // 2   # start in middle

    def reset(self) -> int:
        self.state = self.N // 2
        return self.state

    def step(self, action: int) -> Tuple[int, float, bool]:
        self.state += (1 if action == 1 else -1)
        self.state  = max(0, min(self.N - 1, self.state))
        done        = self.state in (0, self.N - 1)
        reward      = 1.0 if self.state == self.N - 1 else 0.0
        return self.state, reward, done


def collect_episode(env: LinearChainMDP, policy: DiscretePolicy) -> dict:
    """Roll out one episode, return tensors for training."""
    obs_list, act_list, lp_list, rew_list = [], [], [], []
    obs  = env.reset()
    done = False
    while not done:
        obs_t  = torch.tensor([obs], dtype=torch.float32)
        act, lp = policy.act(obs_t)
        next_obs, rew, done = env.step(act.item())
        obs_list.append(obs_t)
        act_list.append(act)
        lp_list.append(lp)
        rew_list.append(rew)
        obs = next_obs
    return {
        # stack keeps last dim: each obs is (1,) → (T, 1). cat would wrongly give (T,).
        "obs":       torch.stack(obs_list),
        "actions":   torch.stack(act_list),
        "log_probs": torch.stack(lp_list),
        "rewards":   torch.tensor(rew_list, dtype=torch.float32),
    }


def a2c_step(
    policy:    DiscretePolicy,
    critic:    ValueNetwork,
    optimizer: torch.optim.Optimizer,
    episode:   dict,
    gamma: float = 0.99,
    lam:   float = 0.95,
    vf_coef:  float = 0.5,
    ent_coef: float = 0.01,
) -> dict:
    obs      = episode["obs"]
    rewards  = episode["rewards"]
    log_probs= episode["log_probs"]
    T        = len(rewards)

    # Values for current states; bootstrap V(s_{T})=0 after last transition
    values    = critic(obs)
    next_vals = torch.cat(
        [values[1:].detach(), torch.zeros(1, device=values.device, dtype=values.dtype)]
    )
    returns = torch.tensor(
        compute_returns(rewards.tolist(), gamma),
        device=values.device,
        dtype=values.dtype,
    )

    adv = compute_gae(rewards, values.detach(), next_vals, gamma, lam)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)   # normalise

    # Recompute log-probs and entropy with current policy
    dist     = policy(obs)
    log_probs_new = dist.log_prob(episode["actions"])

    pg_loss  = reinforce_loss(log_probs_new, adv)
    vf_loss  = vf_coef  * value_loss(values, returns)
    ent_loss = ent_coef * dist.entropy().mean()
    total    = pg_loss + vf_loss - ent_loss

    optimizer.zero_grad()
    total.backward()
    torch.nn.utils.clip_grad_norm_(
        list(policy.parameters()) + list(critic.parameters()), 0.5
    )
    optimizer.step()

    return {
        "total": total.item(), "pg": pg_loss.item(),
        "vf": vf_loss.item(), "entropy": ent_loss.item(),
        "episode_reward": rewards.sum().item(), "episode_len": T,
    }


# ── Training run ──────────────────────────────────────────────────────────
torch.manual_seed(42)
env    = LinearChainMDP(N=6)
policy = DiscretePolicy(obs_dim=1, act_dim=2, hidden=32)
critic = ValueNetwork(obs_dim=1, hidden=32)
optim  = torch.optim.Adam(
    list(policy.parameters()) + list(critic.parameters()), lr=3e-3
)

n_episodes   = 300
log_interval = 100
reward_history = []

for ep in range(1, n_episodes + 1):
    episode = collect_episode(env, policy)
    metrics = a2c_step(policy, critic, optim, episode)
    reward_history.append(metrics["episode_reward"])
    if ep % log_interval == 0:
        avg_r = np.mean(reward_history[-log_interval:])
        print(f"Ep {ep:4d}  avg_reward={avg_r:.3f}  entropy={metrics['entropy']:.3f}")

# Verify the policy learned to go right
obs_t  = torch.tensor([3.0])   # middle of chain
dist   = policy(obs_t)
right_prob = dist.probs[1].item()
print(f"\nP(right | state=3) = {right_prob:.3f}  (should be > 0.7 for a trained policy)")
assert right_prob > 0.6, f"Policy did not learn, P(right)={right_prob:.3f}"
print("A2C training ✓")


Ep  100  avg_reward=1.000  entropy=0.005
Ep  200  avg_reward=1.000  entropy=0.004
Ep  300  avg_reward=1.000  entropy=0.004

P(right | state=3) = 0.970  (should be > 0.7 for a trained policy)
A2C training ✓
